<a href="https://colab.research.google.com/github/anitejpadyana/AI-lab/blob/main/Resolution.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:

import re

def is_variable(term):
    return len(term) == 1 and term.islower()


def parse_literal(lit):
    lit = lit.strip()
    neg = lit.startswith('~')

    if neg:
        lit = lit[1:].strip()

    match = re.match(r"([a-zA-Z0-9_]+)\((.*)\)", lit)
    pred = match.group(1)
    args = [arg.strip() for arg in match.group(2).split(',')]

    return (neg, pred, args)


def format_literal(lit_tuple):
    neg, pred, args = lit_tuple
    prefix = "¬ " if neg else ""
    return f"{prefix}{pred}({', '.join(args)})"


def format_clause(clause):
    if not clause:
        return "{   }  (Empty Clause - Contradiction!)"

    return " ∨ ".join(format_literal(lit) for lit in clause)


def unify(args1, args2):
    if len(args1) != len(args2):
        return None

    subs = {}

    for a1, a2 in zip(args1, args2):
        a1 = subs.get(a1, a1)
        a2 = subs.get(a2, a2)

        if a1 == a2:
            continue
        elif is_variable(a1):
            subs[a1] = a2
        elif is_variable(a2):
            subs[a2] = a1
        else:
            return None

    return subs


def apply_substitution(clause, subs):
    new_clause = []

    for neg, pred, args in clause:
        new_args = [subs.get(arg, arg) for arg in args]
        new_clause.append((neg, pred, new_args))

    return new_clause


def resolve(clause1, clause2):
    for i, (neg1, pred1, args1) in enumerate(clause1):
        for j, (neg2, pred2, args2) in enumerate(clause2):

            if pred1 == pred2 and neg1 != neg2:
                subs = unify(args1, args2)

                if subs is not None:
                    rem1 = clause1[:i] + clause1[i+1:]
                    rem2 = clause2[:j] + clause2[j+1:]

                    resolvent = apply_substitution(
                        rem1 + rem2, subs
                    )

                    unique_resolvent = []

                    for lit in resolvent:
                        if lit not in unique_resolvent:
                            unique_resolvent.append(lit)

                    return unique_resolvent, subs

    return None, None


def main():

    kb_cnf = [
        ("~food(x)", "likes(John, x)"),
        ("food(Apple)",),
        ("food(vegetables)",),
        ("~eats(y, z)", "killed(y)", "food(z)"),
        ("eats(Anil, Peanuts)",),
        ("alive(Anil)",),
        ("~eats(Anil, w)", "eats(Harry, w)"),
        ("killed(g)", "alive(g)"),
        ("~alive(k)", "~killed(k)")
    ]

    parsed_kb = [
        [parse_literal(lit) for lit in clause]
        for clause in kb_cnf
    ]

    negated_goal = [
        parse_literal("~likes(John, Peanuts)")
    ]

    print("\n")
    print("1. CONJUNCTIVE NORMAL FORM (CNF) CLAUSES")
    print("\n")

    labels = ['a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i']

    for lbl, clause in zip(labels, parsed_kb):
        print(f"  {lbl}. {format_clause(clause)}")

    print(
        f"  j. {format_clause(negated_goal)} "
        "[Negated Conclusion]"
    )

    print("\n")
    print("2. RESOLUTION PROOF TREE")


    current_clause = negated_goal
    used_indices = set()

    while True:
        resolved = False

        for idx, kb_clause in enumerate(parsed_kb):

            if idx in used_indices:
                continue

            resolvent, subs = resolve(
                current_clause, kb_clause
            )

            if resolvent is not None:
                used_indices.add(idx)

                sub_str = ", ".join(
                    f"{val}/{var}"
                    for var, val in subs.items()
                )

                sub_display = (
                    f"{{{sub_str}}}" if sub_str else "{}"
                )

                print(
                    f"\n  {format_clause(current_clause):<30} "
                    f"   {format_clause(kb_clause)}"
                )

                print("       \\                              /")
                print(
                    "        \\                            /"
                    f"     Unification: {sub_display}"
                )
                print("         \\                          /")
                print("          v                        v")

                print(f"        {format_clause(resolvent)}")

                current_clause = resolvent
                resolved = True
                break

        if not current_clause:
            print(
                "\nHence proved: likes(John, Peanuts) is TRUE."
            )
            break

        if not resolved:
            print("\nCould not resolve further.")
            break


if __name__ == "__main__":
    main()

print("\n\nAnitej Padyana 1WN24CS042")




1. CONJUNCTIVE NORMAL FORM (CNF) CLAUSES


  a. ¬ food(x) ∨ likes(John, x)
  b. food(Apple)
  c. food(vegetables)
  d. ¬ eats(y, z) ∨ killed(y) ∨ food(z)
  e. eats(Anil, Peanuts)
  f. alive(Anil)
  g. ¬ eats(Anil, w) ∨ eats(Harry, w)
  h. killed(g) ∨ alive(g)
  i. ¬ alive(k) ∨ ¬ killed(k)
  j. ¬ likes(John, Peanuts) [Negated Conclusion]


2. RESOLUTION PROOF TREE

  ¬ likes(John, Peanuts)            ¬ food(x) ∨ likes(John, x)
       \                              /
        \                            /     Unification: {Peanuts/x}
         \                          /
          v                        v
        ¬ food(Peanuts)

  ¬ food(Peanuts)                   ¬ eats(y, z) ∨ killed(y) ∨ food(z)
       \                              /
        \                            /     Unification: {Peanuts/z}
         \                          /
          v                        v
        ¬ eats(y, Peanuts) ∨ killed(y)

  ¬ eats(y, Peanuts) ∨ killed(y)    eats(Anil, Peanuts)
       \  